<a href="https://colab.research.google.com/github/api-sage/diaml_projects/blob/main/project6/pafolabi_DIAML_Assignment6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Data, Inference, and Applied Machine Learning - Assignment 6

In [ ]:
# I import all the tools at the top so I do not have to repeat imports in later cells
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import chi2_contingency
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.linear_model import LogisticRegression, Lasso
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, KFold, TimeSeriesSplit, cross_validate, cross_val_score
from sklearn.metrics import accuracy_score, roc_auc_score, roc_curve, mean_squared_error, r2_score

# I fix the random seed so that my splits give the same result each time I run the notebook
RANDOM_STATE = 42
plt.rcParams["figure.dpi"] = 110

## Setup and data

I use one daily file for six cities in East Africa. The cities are Kigali, Nairobi, Kampala, Dodoma, Bujumbura and Dar es Salaam. Each row is one city on one day. I load the file from the GitHub link for this project. If that link fails, I load the copy of the file that sits next to this notebook.

The file has 9,144 rows, with 1,524 days for each city, from 29 July 2022 to 29 September 2026. The first six days for each city have no air pollution values, so 36 rows have gaps in those columns. I keep those rows for Questions 2 and 3, where I only need the weather columns, and I drop them for Question 4, where I need the pollution columns.

In [ ]:
# I load the daily file and I put the rows in city and date order
data_url = "https://raw.githubusercontent.com/api-sage/diaml_projects/refs/heads/main/project6/east_africa_weather_airquality_daily.csv"
try:
    df = pd.read_csv(data_url, parse_dates=["date"])
except Exception:
    df = pd.read_csv("east_africa_weather_airquality_daily.csv", parse_dates=["date"])
df = df.sort_values(["city", "date"]).reset_index(drop=True)

# I name the groups of columns once so later cells stay short
weather_vars = ["temp_mean", "temp_max", "temp_min", "humidity_mean", "dew_point_mean",
                "pressure_mean", "cloud_cover_mean", "wind_speed_mean", "radiation_mean",
                "precipitation_sum"]
pollutant_vars = ["co", "no2", "so2", "o3", "aod", "dust"]
predictors_16 = weather_vars + pollutant_vars
cities = ["Bujumbura", "Dar es Salaam", "Dodoma", "Kampala", "Kigali", "Nairobi"]

# I show the first rows and a short summary of the file
display(df.head())
summary = pd.DataFrame({
    "item": ["rows", "cities", "first date", "last date", "days per city", "rows with no pm2_5"],
    "value": [len(df), df["city"].nunique(), str(df["date"].min().date()), str(df["date"].max().date()),
              len(df) // df["city"].nunique(), int(df["pm2_5"].isna().sum())]
})
display(summary)

### Question 1 - Nonlinearity (20 points)

#### 1.1 Can you explain what nonlinearity is and why it might be necessary to consider nonlinear relationships between variables?

I understand nonlinearity to mean that the link between two variables cannot be drawn well with one straight line. In a linear link, each extra unit of the input adds the same amount to the output, no matter where I start. In a nonlinear link, that added amount changes with the level of the input. The link can bend, it can rise fast and then flatten out, it can stay flat and then jump after a point, or it can go up and then come down.

A plain daily example helps me. If the air gets a bit more humid on a dry day, rain may still be unlikely. Once humidity and cloud pass a high level, a small extra rise can make rain much more likely. The first step and the last step do not have the same effect, so one straight line gives a poor account of what happens.

I need to allow for this shape in real data for three reasons. First, a straight line fit to a bent link gives biased forecasts. It tends to miss at the ends and in the middle in a clear pattern, and the left over errors, the residuals, still hold the bend that the model missed. Second, I can reach the wrong finding. A curved link can look like no link at all if I only fit a line. A clear case is a link that rises and then falls in the same way on both sides. The straight line slope can come out near zero even though the input tells me a lot about the output. Third, many real processes bend by their nature. Growth slows as it nears a limit, a drug has little effect at a tiny dose and no extra effect past a high dose, and rain needs moist air to cross a level before it starts.

For these reasons I first plot the data and the residuals, and I compare a straight line model with a model that can bend. If the bent model cuts the error on new data by a clear amount, I take the bend as real and I keep it in the model.

#### 1.2 Write down the mathematical equation for a nonlinear model and provide an example of an application where it might be appropriate.

I write a general nonlinear model as

$$y_i = f(x_i; \theta) + \varepsilon_i,$$

where $y_i$ is the output for case $i$, $x_i$ holds the inputs, $\theta$ holds the fixed numbers I have to learn from data, called parameters, $f$ is a function that bends, and $\varepsilon_i$ is the error. The key point is that at least one parameter enters $f$ in a nonlinear way, for example inside an exponent or inside a fraction, so I cannot rewrite the model as a simple weighted sum of known terms.

One concrete form is the growth curve

$$y = a\left(1 - e^{-bx}\right) + \varepsilon, \qquad a > 0, \; b > 0.$$

Here $a$ is the top level that $y$ moves towards and $b$ controls how fast it gets there. The parameter $b$ sits inside the exponential term, which is what makes the model nonlinear in its parameters. I note that a curve such as a squared term, $y = \beta_0 + \beta_1 x + \beta_2 x^2 + \varepsilon$, bends as a plot but stays linear in its parameters. I treat that case as a linear model with a bent input, which is not the same thing as the model above.

A good use of this form is the link between drug dose and drug effect. At a very low dose the effect is near zero. As the dose rises the effect grows fast, and past a high dose the effect flattens near a top level because the body cannot respond more. A straight line would keep rising without end and would at some point predict an effect above the top level that the body can give, which makes no sense. The curve above starts at zero, rises, and flattens at $a$, so it matches the way the body acts and each parameter has a plain meaning I can report.

#### 1.3 Can a nonlinear model be more parsimonious than a linear model? Write down mathematical formulae for both the linear and nonlinear models to support your answer.

Yes. I can give a case where the nonlinear model needs far fewer parameters. By parsimonious I mean a model that fits well with few parameters, so that each parameter earns its place.

Suppose the true link rises fast at first and then flattens near a top level, as in Question 1.2. If I insist on a model that is linear in its parameters, I have to chase the bend with many polynomial terms:

$$y = \beta_0 + \beta_1 x + \beta_2 x^2 + \beta_3 x^3 + \beta_4 x^4 + \beta_5 x^5 + \varepsilon.$$

This model has six parameters, $\beta_0$ to $\beta_5$, and I may need even more terms if the bend is sharp or the range of $x$ is wide. The fit can still wobble near the ends of the range, and the single $\beta$ values have no plain meaning on their own.

The nonlinear model for the same shape is

$$y = a\left(1 - e^{-bx}\right) + \varepsilon,$$

which has only two parameters. The value $a$ is the top level and $b$ is the speed of approach. With two parameters I get the right shape over the whole range, a clearer account of the process, and less risk of fitting noise, since a model with fewer parameters has less room to chase chance wiggles in the sample.

So I conclude that linear in parameters does not always mean simple. When the true shape is a known bend, the nonlinear form can be the more parsimonious choice. In practice I still check both models on data the model has not seen, because a small parameter count only helps if the shape is close to the truth.

#### 1.4 Surrogate data are used for testing for nonlinearity. What characteristics are typically preserved when generating surrogates? Give the names of two surrogate techniques and describe the approaches for implementing them.

I use surrogate data to ask a fair question: could a linear process have made data that looks like mine? I make many fake series that keep the linear facts of my series but break any nonlinear facts. If a nonlinear measure on my real series stands apart from the same measure on the fakes, I have proof of nonlinearity. If it sits inside the fake range, I have no such proof.

When I make surrogates I try to keep these facts of the original series: the length and the sampling steps, the mean and the spread, the power spectrum, which is the same information as the lag correlations, and, for the better methods, the full distribution of values. The null claim I test depends on the method. With plain phase randomisation the null is a linear Gaussian process. With the amplitude adjusted methods the null is wider: a linear process seen through one fixed nonlinear rescaling of the values.

The first technique I name is phase randomisation, also called Fourier transform surrogates. I take the Fourier transform of my series, which splits it into waves with sizes and phases. I keep every wave size exactly as it is and I replace the phases with random phases, drawn in a way that keeps the series real. I then invert the transform. Because I kept the wave sizes, the surrogate has the same power spectrum and the same lag correlations as the original. The values, though, become close to a Gaussian shape, and any phase links that carry nonlinear structure are broken.

The second technique is the iterated amplitude adjusted Fourier transform, or IAAFT. I start from a shuffled copy of my series so the value distribution is exact from the start. I then repeat two steps many times. In one step I impose the original power spectrum by taking the transform, swapping in the original wave sizes, and inverting. In the next step I impose the original distribution by rank ordering: I sort the current values and replace them with the sorted original values in that rank order. After many rounds the series holds both facts well. The simpler AAFT method does these steps once without the iteration, so its spectrum match is less exact.

To use either method I make, say, 99 surrogates, I compute the same nonlinear statistic on the real series and on each surrogate, for example a local forecast error, and I compare. If the real value falls outside the surrogate range, I reject the linear null at about the 1 in 100 level.

#### 1.5 Define information, entropy, and mutual information, and also provide mathematical formulas. Describe how entropy can be used for constructing a feature for measuring regularity and give an example of an application. Explain how mutual information can be used for feature selection and why it might be better than correlation.

I define information first. The information in an event is large when the event is rare and small when the event is common. For an outcome $x$ with probability $p(x)$, I write

$$I(x) = -\log_2 p(x).$$

With log base 2 the unit is bits. An event with probability one half carries 1 bit, while an event with probability one eighth carries 3 bits.

Entropy is the average information in a random variable. For a discrete variable $X$ I write

$$H(X) = -\sum_x p(x) \log_2 p(x).$$

If one outcome is almost sure, entropy is near zero because the next value holds no surprise. If many outcomes are alike in chance, entropy is high. I also use the joint entropy $H(X, Y)$ for a pair and the conditional entropy $H(X \mid Y) = H(X, Y) - H(Y)$, which is the surprise left in $X$ once I know $Y$.

Mutual information is the drop in surprise about one variable when I learn the other. I write it in two equal ways:

$$I(X; Y) = \sum_{x, y} p(x, y) \log_2 \frac{p(x, y)}{p(x)\, p(y)} = H(X) - H(X \mid Y).$$

It is zero only when $X$ and $Y$ are independent, and it grows as the link of any shape gets stronger. The formula is symmetric, so $I(X; Y) = I(Y; X)$.

To build a regularity feature from entropy, I cut a long signal into short windows and I compute an entropy value in each window. A steady, repeating signal gives low entropy because the next value is easy to guess. A messy signal gives high entropy. Forms I can use are sample entropy, which counts how often short patterns that match stay matched when I extend them by one step, and permutation entropy, which looks at the rank order of small groups of values. As an application, I can track the entropy of the gaps between heart beats from an ECG record. A heart under strain often beats in a more fixed, alike way, so the entropy falls. A fall in this feature can flag a patient who needs a closer check, long before a plain average of the same gaps would change.

For feature selection, I score each candidate input by its mutual information with the target and I keep the high scoring inputs. I prefer this to correlation when I suspect bent links, because correlation only sees straight line links. A clear case is $Y = X^2$ with $X$ spread evenly on both sides of zero. The correlation is near zero, so a correlation screen drops $X$, yet $X$ pins down $Y$ apart from sign, and the mutual information is well above zero, so an MI screen keeps it. The same gain holds for threshold links, where the output only moves once the input crosses a level. I stay aware of two limits. MI needs an estimate of probabilities from bins or from near neighbours, so it can shift with those choices and it needs more data than correlation. MI also scores each input alone, so it can keep two inputs that carry the same news. I check for that overlap after the first screen.

### Question 2 - Classification using trees (25 points)

#### 2.1 Decision trees are often used to transform a set of observations into a specific recommended action. Describe the components (nodes, branches) of a decision tree. What is pruning, and why might it be necessary to prune the tree? Why are decision trees an attractive method for classification in practical applications?

A decision tree turns a set of facts about a case into one clear action by asking a chain of yes or no questions. I picture it as an upside down tree that I read from the top down.

The top box is the root node. It holds all the training cases and it asks the first and strongest question, for example: is rain today at most 0.55 mm? Each box that asks a question is an internal node. One node tests one variable against one cut value. The lines that leave a node are the branches. A common tree has two branches per node, one for yes and one for no, and each branch carries the cases that gave that answer. As I follow the branches down, the groups get smaller and purer, which means each group holds more cases of one class. The end boxes are the leaf nodes, also called terminal nodes. A leaf asks no more questions. It gives the final call, such as rain or no rain, often with the share of training cases of each class in that leaf. Staff can then attach an action to the leaf, for example to send a rain alert or to hold it back. The depth of the tree is the longest chain from the root to a leaf.

Pruning means cutting back branches and turning a node into a leaf, even when the cut makes the fit on the training data a bit worse. I need pruning because a tree grown without limit will keep splitting until each leaf holds very few cases. At that point the tree has memorised noise in the training days, such as one odd storm, and it does badly on new days. A huge tree is also hard to read, hard to turn into action, and it can change a lot if the data shift a bit. I can prune while I grow the tree by setting a top depth or a least number of cases per leaf. I can also grow a full tree and then cut it back, as cost complexity pruning does, where a penalty value often called alpha sets how much fit I trade for a smaller size.

I find trees attractive in practice for plain reasons. A manager or a field worker can follow the rule chain and see why the model gave that call, which builds trust. Trees need no scaling of the inputs, they cope with bent links and thresholds without extra work, and they need little data preparation. They also give a quick ranking of which variables matter most. The main weakness I keep in mind is that one tree can be jumpy: a small change in the data can change the top splits. Pruning and checks on new data help me control that risk.

#### 2.2 Suppose an organization has built a rule-based classifier using domain knowledge. After collecting a large amount of data, outline the steps required to improve upon the existing approach by constructing a data-driven classifier. How would you advise testing the validity of the new model?

I would treat the rule system as the mark to beat and I would keep it frozen while I build the new model. My steps are as follows.

First, I write down the exact job: what counts as a positive case, what inputs are known at the time of the call, and what each type of error costs. Second, I build a clean data set by joining past inputs to the known outcomes. I use the same input fields that the rules use, plus any new fields the data now hold. I fix or flag bad rows and I make sure no field leaks the answer, for example a field that is only filled after the outcome is known. Third, I split the data into three parts for training, tuning and final testing. If the cases run in time order, I keep later cases for testing so the test mimics real use. Fourth, I train a data driven classifier on the training part and I tune its settings on the tuning part. Fifth, I compare the new model with the rule system on the same locked test cases, with the same scores that matter for the job, such as accuracy, ROC AUC and the cost of the two error types. Sixth, I study the cases where the rules win. Some will be data faults I must fix. Others may show real gaps where expert rules still add value, in which case I may keep the rules as a fallback for those rare cases. Last, I run the new model in shadow mode next to the live rules, where it makes calls that no one acts on, and only then do I roll it out in small steps with checks for drift.

For validity I advise that the final test cases stay locked until the model and all its settings are fixed, so the test score stays honest. I report a range for each score, not one number, so the reader sees the room for chance. I check the score within key groups, such as each site or each season, and I check that predicted chances match real rates when the action depends on a chance. A strong extra check is a fresh time block or a new site that played no part in training. If the new model only wins on data it has seen, I do not trust it.

#### 2.3 Use the daily weather data for six East African cities (Kigali, Nairobi, Kampala, Dodoma, Bujumbura and Dar es Salaam) provided on Canvas. The target variable is rain_tomorrow, which is 1 if more than 1 mm of rain falls on the next day and 0 otherwise. Report the proportion of days with rain tomorrow in each city. Which city has the highest and which the lowest share of rainy days? Test whether the differences between cities could be due to chance. (Use a chi-square test of independence between city and rain_tomorrow.)

In [ ]:
# I count dry days and rainy days for each city and I work out the rainy share
counts = pd.crosstab(df["city"], df["rain_tomorrow"], margins=False)
counts.columns = ["no_rain_days", "rain_days"]
counts["total_days"] = counts["no_rain_days"] + counts["rain_days"]
counts["rain_share"] = counts["rain_days"] / counts["total_days"]
counts = counts.reindex(cities).sort_values("rain_share", ascending=False)
display(counts.round(4))

# I test if city and rain tomorrow look independent or linked
chi2_stat, p_value, dof, expected = chi2_contingency(pd.crosstab(df["city"], df["rain_tomorrow"]))
test_result = pd.DataFrame({
    "item": ["chi-square statistic", "degrees of freedom", "p-value", "smallest expected count"],
    "value": [round(chi2_stat, 2), dof, p_value, round(expected.min(), 1)]
})
display(test_result)

# I draw the rainy share for each city so the gap is easy to see
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.bar(counts.index, counts["rain_share"] * 100, color="steelblue")
ax.set_ylabel("Days with rain tomorrow (%)")
ax.set_title("Share of days with rain tomorrow, by city")
ax.set_ylim(0, 70)
for i, v in enumerate(counts["rain_share"] * 100):
    ax.text(i, v + 1, f"{v:.1f}%", ha="center", fontsize=9)
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()

The table and the plot give the same story. Kampala has the highest share of rainy days at 60.3 percent, with 919 rainy days out of 1,524. Dar es Salaam comes next at 49.7 percent, then Kigali at 42.5 percent, Nairobi at 40.6 percent and Bujumbura at 34.3 percent. Dodoma has the lowest share at 21.5 percent, with only 328 rainy days. The gap between the wettest and the driest city is close to 39 points, which is large for samples of this size. Each city has the same count of 1,524 days, so the shares can be compared head to head. Across all six cities, 41.5 percent of days are followed by rain.

The chi-square test asks if gaps this large could arise by chance if city and rain tomorrow had no link. The statistic is 549.20 on 5 degrees of freedom, and the p-value is far below 0.001. The smallest expected count under no link is about 632, which is well above the small values that would make the test shaky. I therefore treat the city gaps as real and not as chance noise. In plain terms, where a city sits matters a lot for next day rain: Kampala sits in a wet belt near Lake Victoria, while Dodoma sits in a drier inland zone, so the ranking fits what I expect from the map.

#### 2.4 Pool the observations from all six cities and, using today's ten weather variables as predictors (temp_mean, temp_max, temp_min, humidity_mean, dew_point_mean, pressure_mean, cloud_cover_mean, wind_speed_mean, radiation_mean and precipitation_sum), construct a decision tree to classify rain_tomorrow and display the structure of the tree using a graphic.

In [ ]:
# I pool all cities and I fit a tree with no size limit first
X_all = df[weather_vars]
y_all = df["rain_tomorrow"]

full_tree = DecisionTreeClassifier(random_state=RANDOM_STATE)
full_tree.fit(X_all, y_all)

size_table = pd.DataFrame({
    "item": ["depth of full tree", "leaves in full tree", "days used", "predictors used"],
    "value": [full_tree.get_depth(), full_tree.get_n_leaves(), len(df), len(weather_vars)]
})
display(size_table)

# The full tree is far too big to read, so I draw only its top three levels
fig, ax = plt.subplots(figsize=(16, 7))
plot_tree(full_tree, max_depth=3, feature_names=weather_vars,
          class_names=["no rain", "rain"], filled=True, rounded=True, fontsize=7, ax=ax)
ax.set_title("Top three levels of the full tree (the full tree has depth 30)")
plt.tight_layout()
plt.show()

I fit the first tree with no size limit on all 9,144 pooled days. It grows to depth 30 with 1,623 leaves. That size tells me at once that the raw tree has memorised the training days in fine detail and that no person could read it as a rule set. The graphic therefore shows only the top three levels, which hold the splits that matter most.

The root split is rain today: days with precipitation_sum at most 0.55 mm go one way and wetter days go the other. That split makes sense to me because rain tends to come in spells, so a wet day raises the chance of a wet next day. The next splits use humidity_mean, which also fits my expectation, since moist air is needed for rain to keep forming. I read one path as follows: a day with more than 0.55 mm of rain and high humidity falls in a part of the tree where rain leaves dominate, while a dry day with low humidity falls where dry leaves dominate. The full tree adds hundreds of small splits below this level. Most of those splits chase noise rather than signal, which is the problem I address next with pruning.

#### 2.5 Evaluate the performance of the tree before and after pruning, using 10-fold cross-validation. Prune the tree either by limiting its depth or using cost-complexity pruning, and choose the pruning strength (maximum depth or alpha) by cross-validation.

In [ ]:
# I score each tree size with 10-fold cross-validation and I keep the train score to show overfitting
cv_class = StratifiedKFold(n_splits=10, shuffle=True, random_state=RANDOM_STATE)

depth_list = [1, 2, 3, 4, 5, 6, 8, 10]
depth_rows = []
for depth in depth_list:
    tree = DecisionTreeClassifier(max_depth=depth, random_state=RANDOM_STATE)
    scores = cross_validate(tree, X_all, y_all, cv=cv_class,
                            scoring=["accuracy", "roc_auc"], return_train_score=True)
    depth_rows.append({"max_depth": depth,
                       "accuracy": scores["test_accuracy"].mean(),
                       "roc_auc": scores["test_roc_auc"].mean(),
                       "train_roc_auc": scores["train_roc_auc"].mean()})

# I add the full tree as the before pruning case
scores_full = cross_validate(DecisionTreeClassifier(random_state=RANDOM_STATE),
                             X_all, y_all, cv=cv_class,
                             scoring=["accuracy", "roc_auc"], return_train_score=True)
depth_rows.append({"max_depth": "no limit",
                   "accuracy": scores_full["test_accuracy"].mean(),
                   "roc_auc": scores_full["test_roc_auc"].mean(),
                   "train_roc_auc": scores_full["train_roc_auc"].mean()})
depth_table = pd.DataFrame(depth_rows)
display(depth_table.round(4))

# I draw accuracy and ROC AUC against depth
plot_rows = depth_table[depth_table["max_depth"] != "no limit"].copy()
plot_rows["max_depth"] = plot_rows["max_depth"].astype(int)
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(plot_rows["max_depth"], plot_rows["accuracy"], marker="o", label="accuracy")
ax.plot(plot_rows["max_depth"], plot_rows["roc_auc"], marker="s", label="ROC AUC")
ax.set_xlabel("Maximum depth")
ax.set_ylabel("Cross-validation score")
ax.set_title("Tree score by maximum depth (10-fold cross-validation)")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# I also try cost complexity pruning so I can compare the two pruning styles
alpha_list = [0.0, 0.0005, 0.001, 0.0015, 0.002, 0.003, 0.005, 0.01]
alpha_rows = []
for alpha in alpha_list:
    tree = DecisionTreeClassifier(ccp_alpha=alpha, random_state=RANDOM_STATE)
    scores = cross_validate(tree, X_all, y_all, cv=cv_class, scoring=["accuracy", "roc_auc"])
    alpha_rows.append({"alpha": alpha,
                       "accuracy": scores["test_accuracy"].mean(),
                       "roc_auc": scores["test_roc_auc"].mean()})
alpha_table = pd.DataFrame(alpha_rows)
display(alpha_table.round(4))

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(alpha_table["alpha"], alpha_table["accuracy"], marker="o", label="accuracy")
ax.plot(alpha_table["alpha"], alpha_table["roc_auc"], marker="s", label="ROC AUC")
ax.set_xlabel("Alpha (cost complexity penalty)")
ax.set_ylabel("Cross-validation score")
ax.set_title("Tree score by alpha (10-fold cross-validation)")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# I fix the final tree at depth 4 and I draw it in full
final_tree = DecisionTreeClassifier(max_depth=4, random_state=RANDOM_STATE)
final_tree.fit(X_all, y_all)
final_size = pd.DataFrame({
    "item": ["depth of final tree", "leaves in final tree"],
    "value": [final_tree.get_depth(), final_tree.get_n_leaves()]
})
display(final_size)
fig, ax = plt.subplots(figsize=(18, 8))
plot_tree(final_tree, feature_names=weather_vars, class_names=["no rain", "rain"],
          filled=True, rounded=True, fontsize=7, ax=ax)
ax.set_title("Final pruned tree with maximum depth 4")
plt.tight_layout()
plt.show()

Before pruning, the full tree scores 0.657 on accuracy and 0.647 on ROC AUC in 10-fold cross-validation. Its training ROC AUC is near 1.0. That wide gap between the training score and the test score is the clear sign of overfitting: the tree fits the training days almost in full but loses a lot on days it has not seen. A useful guide is that a model that always says no rain scores 0.585 on accuracy here, since 58.5 percent of days stay dry. The full tree beats that guide by only a small margin once I test it fairly.

Pruning lifts the score a lot. At maximum depth 2 the ROC AUC jumps to about 0.777, at depth 3 to about 0.792, and at depth 4 to 0.797 with accuracy 0.726. Depth 5 is much the same, with ROC AUC 0.797, and past that point the scores fall as the tree starts to memorise again. Cost complexity pruning tells the same story: the best alpha I tried is 0.001, with accuracy 0.729 and ROC AUC 0.795, close to the depth 4 result. Since the two styles agree, I pick the depth limit because it is easier to state and to draw.

My final tree has depth 4 and 16 leaves. Pruning cuts the leaf count from 1,623 to 16 and at the same time lifts accuracy by about 7 points and ROC AUC by about 15 points. That trade is the heart of this part: I give up a near perfect fit on old days and I gain a small rule set that holds up on new days. In the final tree, every path that starts with a dry day ends in a no rain leaf, and rain calls only appear after a wet day with high humidity. I find that rule plain enough to act on.

#### 2.6 Compare the performance of the final tree with logistic regression using accuracy and ROC AUC, and comment on the advantages and disadvantages of both. Which weather variables does the tree rely on most? Which model would you use for a daily rain classification, and why?

In [ ]:
# I compare the final tree with logistic regression on the same folds
logreg = Pipeline([("scaler", StandardScaler()),
                   ("model", LogisticRegression(max_iter=1000))])
models = {"Final tree (depth 4)": DecisionTreeClassifier(max_depth=4, random_state=RANDOM_STATE),
          "Logistic regression": logreg}
compare_rows = []
for name, model in models.items():
    scores = cross_validate(model, X_all, y_all, cv=cv_class, scoring=["accuracy", "roc_auc"])
    compare_rows.append({"model": name,
                         "accuracy_mean": scores["test_accuracy"].mean(),
                         "accuracy_std": scores["test_accuracy"].std(),
                         "roc_auc_mean": scores["test_roc_auc"].mean(),
                         "roc_auc_std": scores["test_roc_auc"].std()})
compare_table = pd.DataFrame(compare_rows)
display(compare_table.round(4))

# I collect out of fold chances from both models and I draw the two ROC curves
prob_tree = np.zeros(len(df))
prob_logreg = np.zeros(len(df))
for train_idx, test_idx in cv_class.split(X_all, y_all):
    tree_fold = DecisionTreeClassifier(max_depth=4, random_state=RANDOM_STATE)
    tree_fold.fit(X_all.iloc[train_idx], y_all.iloc[train_idx])
    prob_tree[test_idx] = tree_fold.predict_proba(X_all.iloc[test_idx])[:, 1]
    logreg_fold = Pipeline([("scaler", StandardScaler()),
                            ("model", LogisticRegression(max_iter=1000))])
    logreg_fold.fit(X_all.iloc[train_idx], y_all.iloc[train_idx])
    prob_logreg[test_idx] = logreg_fold.predict_proba(X_all.iloc[test_idx])[:, 1]

fig, ax = plt.subplots(figsize=(6.5, 5.5))
for probs, label in [(prob_tree, "Final tree"), (prob_logreg, "Logistic regression")]:
    fpr, tpr, _ = roc_curve(y_all, probs)
    ax.plot(fpr, tpr, label=f"{label} (AUC = {roc_auc_score(y_all, probs):.3f})")
ax.plot([0, 1], [0, 1], linestyle="--", color="grey", label="Chance")
ax.set_xlabel("False positive rate")
ax.set_ylabel("True positive rate")
ax.set_title("ROC curves from out of fold predictions")
ax.legend()
plt.tight_layout()
plt.show()

# I show how much the final tree leans on each weather variable
importance = pd.DataFrame({"variable": weather_vars,
                           "importance": final_tree.feature_importances_}).sort_values("importance", ascending=True)
display(importance.round(4))
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.barh(importance["variable"], importance["importance"], color="steelblue")
ax.set_xlabel("Share of split gain")
ax.set_title("Variable importance in the final tree")
plt.tight_layout()
plt.show()

Logistic regression comes out a bit ahead on both scores. It reaches accuracy 0.734 and ROC AUC 0.810, against 0.726 and 0.797 for the final tree. The gap is small next to the fold to fold spread of about 0.015, so I read it as a slight edge rather than a rout. The ROC plot shows the same thing: the logistic curve sits a little above the tree curve across most thresholds. Both models beat the always dry guide of 0.585 on accuracy by a wide margin, so both carry real signal. The out of fold AUC values on the plot match the cross-validation table, which gives me trust in the comparison.

The tree leans most on rain today. Precipitation_sum holds about 0.80 of the split gain, and humidity_mean holds about 0.15. Every other variable holds less than 0.02 each, and temp_mean, temp_max and wind_speed_mean hold zero in this pruned tree. That does not mean the other variables carry no news. It means that once the tree knows rain today and humidity, the extra splits add too little to survive pruning. The tree story is one of persistence: wet and moist days tend to stay wet.

Each model has a clear strength and a clear cost. The tree gives me rules I can print and act on, it needs no scaling, and it finds thresholds on its own. Its cost is that its calls jump at cut values, it can change if the data shift, and it wastes small steady effects that do not win a split. Logistic regression gives me a smooth chance for each day, it uses all ten variables in small steps, and it ranks days a bit better. Its costs are that it needs scaled inputs, it draws a straight boundary in the scaled space unless I add bent terms by hand, and its coefficients are less direct as field rules than a tree path.

For a daily rain classification I would use logistic regression. My reason is that a daily service needs a chance, not just a hard call, so that staff can set their own alert level for the cost of a missed rainy day against a false alarm. Logistic regression gives the better ranking and the smoother chance at no loss in accuracy. I would keep the pruned tree next to it as the plain language account of why wet and moist days score high.

### Question 3 - Classification using KNN: will it rain tomorrow? (25 points)

#### 3.1 By focusing on small neighborhoods of state space, it is possible to construct parsimonious models. Describe the concept behind this general approach and a step-by-step procedure for implementing such a model.

The core idea I use is that days with near alike weather tend to have near alike next days. I place each day as a point in a state space, where the axes are the weather values for that day, such as temperature, humidity and rain today. A small neighbourhood is the set of past days whose points fall close to the point for a new day. Instead of fitting one global equation for all days, I let the near past days speak for the new day. I call this view parsimonious because I need very few global fixed numbers. My main choices are how I measure closeness and how many near days I listen to. The data store itself replaces a long list of fitted coefficients.

My step by step plan for such a model is as follows. First, I build one state vector per day from the inputs that are known at forecast time. Second, I put the inputs on a fair scale and I pick a distance rule, since closeness has no meaning while one axis is in hundreds and another is in single digits. Third, I pick the neighbourhood size, the count $k$ of near days, by cross-validation rather than by taste. Fourth, for a new day I find its $k$ nearest past days. Fifth, I combine their outcomes: for a class I take a vote, and the vote share gives a rough chance, while for a number I take an average. Sixth, I score the rule on days the rule has not seen and I adjust $k$, the scale and the distance until the held out score stops improving.

Two trade offs guide my choices. A tiny neighbourhood follows local detail but jumps with noise, since one odd day can flip the call. A wide neighbourhood is steady but can blur real local patterns, such as a coastal pattern washed out by inland days. I also keep in mind that as I add more axes, all days start to look far apart, so local methods need care with the input list.

#### 3.2 Consider the challenge of predicting whether it will rain tomorrow, using today's weather as predictors. To construct a KNN classifier, how will you transform the available variables?

I pool the six cities and I use the same ten weather variables as in Question 2, so the KNN result can be compared with the tree on fair terms. I do not add city as an input, since the question frames the task with today's weather as the predictors. All ten inputs are numbers, so I need no dummy variables. The weather columns have no gaps, so I need no filling rule. The target, rain_tomorrow, stays out of the input set.

My main transformation is standardization, also called scaling to z-scores. For each variable I subtract its mean and divide by its standard deviation, so each variable ends with mean zero and spread one. I fit those means and spreads on the training folds only, then I apply the same numbers to the test fold. That order matters: if I scale with the full data, facts from the test days leak into training. The table below shows why I must scale. Pressure runs in the hundreds, radiation also runs in the hundreds, humidity runs in tens, and rain today runs mostly from zero to a few units. With raw values, a distance sum is ruled by pressure and radiation, and humidity or rain can barely move the total, even though rain today was the strongest signal in the tree.

I keep two further points in mind. Rain today has a skewed shape, with many zeros and a long tail of wet days, so very wet days sit far from the rest even after scaling. A log style squeeze would be a fair variant to test, but I keep the raw scale here so the input set stays the same as the tree task. The three temperature variables move together, so distance counts heat more than once. I keep all three for a fair match with Question 2, and I note that dropping repeats is an option if I want a leaner state space.

In [ ]:
# I show the range of each weather variable to explain why scaling is needed
scale_table = df[weather_vars].agg(["min", "max", "mean", "std"]).T
scale_table = scale_table.rename(columns={"std": "standard_deviation"})
display(scale_table.round(2))

#### 3.3 Calculate the performance of the classifier versus the number of neighbors used and provide a graphic to display the result. What is the optimal number of neighbors based on ROC AUC using cross-validation of 10 folds?

In [ ]:
# I try many neighbourhood sizes with scaled inputs and Euclidean distance
cv_class = StratifiedKFold(n_splits=10, shuffle=True, random_state=RANDOM_STATE)
k_list = [1, 3, 5, 7, 9, 11, 15, 21, 31, 41, 51, 61, 75, 91, 101, 151]
k_rows = []
for k in k_list:
    knn = Pipeline([("scaler", StandardScaler()),
                    ("model", KNeighborsClassifier(n_neighbors=k))])
    scores = cross_validate(knn, X_all, y_all, cv=cv_class, scoring=["accuracy", "roc_auc"])
    k_rows.append({"k": k,
                   "accuracy": scores["test_accuracy"].mean(),
                   "roc_auc": scores["test_roc_auc"].mean(),
                   "roc_auc_std": scores["test_roc_auc"].std()})
k_table = pd.DataFrame(k_rows)
display(k_table.round(4))

# I draw both scores against the neighbourhood size
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(k_table["k"], k_table["accuracy"], marker="o", label="accuracy")
ax.plot(k_table["k"], k_table["roc_auc"], marker="s", label="ROC AUC")
ax.set_xlabel("Number of neighbours (k)")
ax.set_ylabel("Cross-validation score")
ax.set_title("KNN score by neighbourhood size (scaled, Euclidean, 10-fold)")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

The plot shows the pattern I expect from a local method. With one neighbour the ROC AUC is only about 0.66, because the call hangs on a single past day and jumps with noise. The score climbs fast as I add neighbours: it passes 0.79 near $k = 9$ and 0.80 near $k = 15$. Past that point the gains slow down. The curve flattens into a wide top band. The highest value lands at $k = 91$ with ROC AUC 0.814 and accuracy 0.735, and $k = 75$ and $k = 101$ sit within 0.001 of it. Values from about $k = 51$ to $k = 151$ all sit in that same flat band and within the fold spread, so I read the top as a range rather than a magic number.

My optimal choice on ROC AUC is therefore $k = 91$ with scaled inputs and Euclidean distance, with the clear note that any $k$ in the flat top band serves much the same. I refine the distance rule in the next part, and that search sets the final KNN I carry forward. I like that the optimum is wide: it means the method is not fussy, and a nearby $k$ would serve as well. Accuracy tells the same story, rising from 0.67 at $k = 1$ to about 0.735 at the top. The lesson I take is that one day of weather is a noisy guide, while the vote of dozens of alike past days steadies the call without washing out the local pattern.

#### 3.4 Explain why some distance metrics are sensitive to the kind of features used. Evaluate the performance using 4 different distance metrics (e.g., Euclidean, Manhattan, and Chebyshev, and cosine), with and without standardizing the predictors.

A distance is a sum over features, so the units and spread of each feature set its weight in that sum. Euclidean distance squares each gap before it adds, so a feature with large units, such as pressure in the hundreds, can swamp a feature with small units, such as rain in single digits, even when the small feature carries the signal. Manhattan distance adds plain gaps, so it is less ruled by one large gap, but it still counts a gap of 50 pressure units as far larger than a gap of 2 rain units. Chebyshev distance takes only the largest single gap, so the whole call can hang on whichever feature has the widest raw range. Cosine distance compares the angle or pattern of the two vectors and ignores their overall size, yet the angle still bends toward the large unit axes when I feed raw values. Standardizing puts each feature in spread units, so each axis starts with an equal say. That is why I expect scaling to help the first three metrics a lot, and cosine to a lesser but still real degree.

In the code I hold the task fixed and I vary only the metric and the scaling. For each of the eight mixes I search the same $k$ grid and I keep the mix at its own best $k$ on ROC AUC, so each metric gets a fair chance.

In [ ]:
# I compare four distance rules with and without scaling, each at its best k
metric_grid = {"euclidean": {"metric": "euclidean"},
               "manhattan": {"metric": "manhattan"},
               "chebyshev": {"metric": "chebyshev"},
               "cosine": {"metric": "cosine"}}
k_short = [5, 11, 21, 31, 51, 75, 101]
metric_rows = []
for scaled in [True, False]:
    for metric_name, metric_args in metric_grid.items():
        best = None
        for k in k_short:
            steps = []
            if scaled:
                steps.append(("scaler", StandardScaler()))
            steps.append(("model", KNeighborsClassifier(n_neighbors=k, **metric_args)))
            scores = cross_validate(Pipeline(steps), X_all, y_all, cv=cv_class,
                                    scoring=["accuracy", "roc_auc"])
            auc = scores["test_roc_auc"].mean()
            if best is None or auc > best["roc_auc"]:
                best = {"scaled": "yes" if scaled else "no", "metric": metric_name, "best_k": k,
                        "accuracy": scores["test_accuracy"].mean(), "roc_auc": auc}
        metric_rows.append(best)
metric_table = pd.DataFrame(metric_rows).sort_values("roc_auc", ascending=False)
display(metric_table.round(4))

# I draw ROC AUC for each metric with and without scaling
pivot = metric_table.pivot(index="metric", columns="scaled", values="roc_auc").reindex(
    ["manhattan", "euclidean", "cosine", "chebyshev"])
fig, ax = plt.subplots(figsize=(8, 4.5))
x = np.arange(len(pivot))
ax.bar(x - 0.2, pivot["yes"], width=0.4, label="scaled")
ax.bar(x + 0.2, pivot["no"], width=0.4, label="raw")
ax.set_xticks(x)
ax.set_xticklabels(pivot.index)
ax.set_ylabel("Best ROC AUC")
ax.set_title("Best KNN ROC AUC by distance metric and scaling")
ax.set_ylim(0.7, 0.85)
ax.legend()
plt.tight_layout()
plt.show()

Scaling helps every metric, which backs the point above that raw units distort closeness. My best mix overall is Manhattan distance with scaling, at ROC AUC 0.816, with Euclidean and scaling a hair behind at 0.813. Cosine with scaling comes next, and Chebyshev trails the pack, which fits its habit of letting one axis decide. Without scaling, Manhattan still leads its group, at about 0.807, because plain gaps resist the huge squared pressure gaps that hurt Euclidean. The raw Chebyshev score is the weakest, at about 0.789.

Two lessons stand out for me. First, the choice to scale matters more than the choice among the top metrics: the scaled Manhattan edge over scaled Euclidean is tiny, while the gain from scaling itself is clear for Chebyshev, cosine and Euclidean. Second, Manhattan earns its lead in a plausible way. Weather gaps are often sharp in one axis, such as a sudden rain jump, and a sum of plain gaps is less thrown by one extreme axis than a sum of squares. I carry Manhattan with scaling at its best $k = 101$ forward as my best KNN, while I note that Euclidean at the same scale would serve much the same.

#### 3.5 Compare the best KNN classifier with the final decision tree from Question 2 using accuracy and ROC AUC, and comment on the advantages and disadvantages of KNN.

In [ ]:
# I compare my best KNN with the final tree on the same folds
best_knn = Pipeline([("scaler", StandardScaler()),
                     ("model", KNeighborsClassifier(n_neighbors=101, metric="manhattan"))])
knn_tree_rows = []
for name, model in {"Best KNN (Manhattan, k=101)": best_knn,
                    "Final tree (depth 4)": DecisionTreeClassifier(max_depth=4, random_state=RANDOM_STATE)}.items():
    scores = cross_validate(model, X_all, y_all, cv=cv_class, scoring=["accuracy", "roc_auc"])
    knn_tree_rows.append({"model": name,
                          "accuracy": scores["test_accuracy"].mean(),
                          "roc_auc": scores["test_roc_auc"].mean()})
display(pd.DataFrame(knn_tree_rows).round(4))

# I draw the ROC curves from out of fold predictions for both models
prob_knn = np.zeros(len(df))
for train_idx, test_idx in cv_class.split(X_all, y_all):
    fold_model = Pipeline([("scaler", StandardScaler()),
                           ("model", KNeighborsClassifier(n_neighbors=101, metric="manhattan"))])
    fold_model.fit(X_all.iloc[train_idx], y_all.iloc[train_idx])
    prob_knn[test_idx] = fold_model.predict_proba(X_all.iloc[test_idx])[:, 1]

fig, ax = plt.subplots(figsize=(6.5, 5.5))
for probs, label in [(prob_knn, "Best KNN"), (prob_tree, "Final tree"), (prob_logreg, "Logistic regression")]:
    fpr, tpr, _ = roc_curve(y_all, probs)
    ax.plot(fpr, tpr, label=f"{label} (AUC = {roc_auc_score(y_all, probs):.3f})")
ax.plot([0, 1], [0, 1], linestyle="--", color="grey", label="Chance")
ax.set_xlabel("False positive rate")
ax.set_ylabel("True positive rate")
ax.set_title("ROC curves from out of fold predictions")
ax.legend()
plt.tight_layout()
plt.show()

My best KNN edges out the final tree on both scores: accuracy 0.739 against 0.726, and ROC AUC 0.816 against 0.797. The ROC plot puts KNN close to logistic regression and a clear step above the tree curve. The edge is modest, about two points of AUC, so I read KNN as the better ranker here, not as a different class of model. All three models sit in a tight band, which tells me the ten weather variables hold a limited amount of next day signal and each method extracts most of it.

The gains of KNN are that it makes no guess about the shape of the boundary, so bent and threshold patterns come free, it uses all ten variables in a smooth way, and adding new days to the store updates it at once with no refit. Its costs are plain too. It needs scaled inputs and a stored copy of the past days, each new forecast needs a search through the store, and the two knobs of $k$ and distance must be tuned with care. It also gives no printed rule: I can show the near days behind a call, but I cannot hand staff a short rule list as I can with the tree. With many axes its edge would fade, since near and far lose meaning, though with ten axes it still works well. On balance I rate KNN the stronger pure scorer in this pooled task, and the tree the stronger tool when a plain account of the call matters most.

#### 3.6 The data are consecutive days. Explain why random cross-validation might overestimate performance when forecasting future rainfall. Suggest an alternative validation approach. Which model would you use for a daily rain forecast, and why?

Random cross-validation shuffles days before it splits them, so the training set holds days that sit right next to each test day in time. Weather comes in spells that last for days. If a wet spell runs from Monday to Friday, a shuffled split can put Monday and Friday in training and Wednesday in test. The model then gets credit for spotting a spell it has half seen, which is far easier than calling a fresh spell next month. Rain today, the strongest input, makes this leak worse, since its value in a test day is near copied in the train days around it. Pooling six cities adds a second leak: the same season and the same large weather systems appear on both sides of the split. The result is a score that flatters the model for the real job, where each forecast must face days that all lie ahead.

My alternative is to validate in time order. I sort each city by date and I always test on a later block than I train on. One sound form is forward chaining, where the training window grows and the test block steps ahead. Another is a single honest holdout, such as the last fifth of days in each city, kept aside until the end. I also leave the tuning inside the past block, so no future day shapes any choice. A short gap between train and test helps break the spell overlap. In the code I run both checks: a time series split on the pooled days, and a last block holdout city by city.

In [ ]:
# I recheck the three models with splits that respect time order
time_cv = TimeSeriesSplit(n_splits=5)
pool_sorted = df.sort_values("date").reset_index(drop=True)
X_time = pool_sorted[weather_vars]
y_time = pool_sorted["rain_tomorrow"]
time_models = {"Best KNN (Manhattan, k=101)": Pipeline([("scaler", StandardScaler()),
                                                       ("model", KNeighborsClassifier(n_neighbors=101, metric="manhattan"))]),
               "Final tree (depth 4)": DecisionTreeClassifier(max_depth=4, random_state=RANDOM_STATE),
               "Logistic regression": Pipeline([("scaler", StandardScaler()),
                                                ("model", LogisticRegression(max_iter=1000))])}
time_rows = []
for name, model in time_models.items():
    scores = cross_validate(model, X_time, y_time, cv=time_cv, scoring=["accuracy", "roc_auc"])
    time_rows.append({"model": name, "time_accuracy": scores["test_accuracy"].mean(),
                      "time_roc_auc": scores["test_roc_auc"].mean()})
display(pd.DataFrame(time_rows).round(4))

# I hold out the last fifth of days in each city and I train only on the earlier days
holdout_rows = []
for city in cities:
    sub = df[df["city"] == city].sort_values("date")
    cut = int(len(sub) * 0.8)
    train_part = sub.iloc[:cut]
    test_part = sub.iloc[cut:]
    scaler = StandardScaler().fit(train_part[weather_vars])
    knn_hold = KNeighborsClassifier(n_neighbors=101, metric="manhattan")
    knn_hold.fit(scaler.transform(train_part[weather_vars]), train_part["rain_tomorrow"])
    chances = knn_hold.predict_proba(scaler.transform(test_part[weather_vars]))[:, 1]
    calls = knn_hold.predict(scaler.transform(test_part[weather_vars]))
    holdout_rows.append({"city": city, "test_days": len(test_part),
                         "accuracy": accuracy_score(test_part["rain_tomorrow"], calls),
                         "roc_auc": roc_auc_score(test_part["rain_tomorrow"], chances)})
display(pd.DataFrame(holdout_rows).round(4))

The time aware checks cool the scores down, as I feared. On the pooled time series split, KNN falls from 0.816 under random folds to 0.811, and the tree falls from 0.797 to about 0.786. Logistic regression holds steady at about 0.812 and moves into first place. The city holdouts show a wider spread: the same KNN scores 0.91 on ROC AUC for the last block in Dodoma, but only 0.73 in Dar es Salaam and 0.70 in Kampala. That spread warns me that one pooled random score hides how much the task shifts by place and by season. The drops are not huge, so the models do carry forward in time, but the random fold ranking was too kind to the local method, which gains most from near twin days.

For a daily rain forecast I would use logistic regression. It keeps the best score when the test days lie ahead, it gives a smooth chance that staff can threshold by cost, it refits fast each day, and it does not need a search through stored days for each call. KNN stays a close second and a good check, since its local view can flag regimes the straight boundary misses. The pruned tree stays my explainer. My order for live use is logistic regression first, KNN as a challenger scored on each new month, and the tree as the plain account of the call.